# SOLUCIÓN – Ejercicio alumnos: Breast Cancer

**Uso exclusivo del docente.**  
Contiene las resoluciones esperadas de todas las celdas TODO y respuestas modelo a las preguntas de reflexión.

## 0. Carga de librerías y datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
df = X.copy()
df["target"] = y
df["diagnostico"] = df["target"].map({0: "malignant", 1: "benign"})

print("Forma:", X.shape)
print("Clases:", data.target_names)
print("\nDistribución:")
print(df["diagnostico"].value_counts())
df.head()

## Parte 1 – Visualización

### 1.1 Distribución de clases (solución)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df["diagnostico"].value_counts()
# Ordenamos para que malignant quede primero (más relevante clínicamente)
counts = counts.reindex(["malignant", "benign"])

bars = ax.bar(counts.index, counts.values, color=["#8B4513", "#A8864A"], edgecolor="white", width=0.6)
ax.set_title("Distribución de diagnósticos – Breast Cancer Wisconsin")
ax.set_ylabel("Cantidad de muestras")
ax.set_xlabel("Diagnóstico")
ax.set_ylim(0, counts.max() * 1.15)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val),
            ha="center", va="bottom", fontweight="bold", fontsize=12)

plt.tight_layout()
plt.show()

print(f"Proporción malignant: {counts['malignant']/len(df):.1%}")
print(f"Proporción benign:    {counts['benign']/len(df):.1%}")
print("→ Hay un leve desbalance a favor de benign. Por eso usaremos StratifiedKFold y F1.")

### 1.2 Boxplots (solución)

In [ ]:
features = ["mean radius", "worst concave points"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, feat in zip(axes, features):
    sns.boxplot(
        data=df, x="diagnostico", y=feat, ax=ax,
        hue="diagnostico", legend=False,
        palette={"malignant": "#8B4513", "benign": "#A8864A"},
        order=["malignant", "benign"]
    )
    ax.set_title(f"Distribución de '{feat}' por diagnóstico")
    ax.set_xlabel("")

plt.suptitle("Comparación de características entre clases", y=1.02)
plt.tight_layout()
plt.show()

print("Observación esperada:")
print("- 'mean radius' y especialmente 'worst concave points' muestran separación clara.")
print("- Los tumores malignos tienden a tener radio mayor y más puntos cóncavos.")

### 1.3 Scatter plot (solución)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="mean radius",
    y="worst concave points",
    hue="diagnostico",
    palette={"malignant": "#8B4513", "benign": "#A8864A"},
    s=60, alpha=0.75, ax=ax
)
ax.set_title("Tumores malignos tienden a mayor radio y más puntos cóncavos")
ax.set_xlabel("Mean radius")
ax.set_ylabel("Worst concave points")
plt.legend(title="Diagnóstico")
plt.tight_layout()
plt.show()

print("Principios aplicados:")
print("- Color con significado (marrón = malignant, dorado = benign)")
print("- Título que comunica el insight (no solo 'Scatter plot')")
print("- Similitud (Gestalt): puntos del mismo color se agrupan visualmente")

## Parte 2 – Validación Cruzada

### 2.1 Variabilidad de una sola partición (solución)

In [ ]:
model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)

accuracies = []
print("Accuracy en 8 divisiones aleatorias (test_size=0.25):\n")
for i in range(8):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=i, stratify=y
    )
    model.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te))
    accuracies.append(acc)
    print(f"  random_state={i} → Accuracy: {acc:.4f}")

print(f"\nPromedio: {np.mean(accuracies):.4f}")
print(f"Desviación estándar: {np.std(accuracies):.4f}")
print(f"Rango: {np.min(accuracies):.4f} – {np.max(accuracies):.4f}")
print("\n→ Incluso con stratify, el accuracy varía. Una sola partición puede engañarnos.")

### 2.2 Stratified K-Fold (solución)

In [ ]:
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skfold, scoring="accuracy")

print("=== Stratified K-Fold (5 folds) ===")
print("Scores por fold:", np.round(scores, 4))
print(f"Promedio: {scores.mean():.4f}  ±  {scores.std():.4f}")
print("\n→ Estimación más estable y representativa del desempeño real.")

## Parte 3 – GridSearchCV

### 3.1 GridSearchCV (solución)

In [ ]:
param_grid = {
    "n_estimators": [40, 80, 120],
    "max_depth": [3, 5, 8, None],
    "min_samples_split": [2, 5]
}

print("Combinaciones:", 
      len(param_grid["n_estimators"]) * 
      len(param_grid["max_depth"]) * 
      len(param_grid["min_samples_split"]))

grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid.fit(X, y)

print("\n=== Resultados ===")
print("Mejores hiperparámetros:", grid.best_params_)
print(f"Mejor F1 (CV): {grid.best_score_:.4f}")

In [ ]:
# Extra: ver top 5
results = pd.DataFrame(grid.cv_results_)
top5 = (results[["params", "mean_test_score", "std_test_score"]]
        .sort_values("mean_test_score", ascending=False)
        .head(5))
print("Top 5 combinaciones por F1:")
for _, row in top5.iterrows():
    print(f"  {row['params']} → F1: {row['mean_test_score']:.4f} (±{row['std_test_score']:.4f})")

### 3.2 Hold-out final (solución del desafío)

In [ ]:
# 1. Separar hold-out
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. GridSearch solo sobre train
grid_final = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    n_jobs=-1
)
grid_final.fit(X_train, y_train)

print("Mejores parámetros (train):", grid_final.best_params_)
print(f"Mejor F1 CV (train): {grid_final.best_score_:.4f}")

# 3. Evaluación única en hold-out
y_pred = grid_final.best_estimator_.predict(X_holdout)
print(f"\nAccuracy hold-out: {accuracy_score(y_holdout, y_pred):.4f}")
print(f"F1 hold-out:       {f1_score(y_holdout, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_holdout, y_pred, target_names=data.target_names))

In [ ]:
cm = confusion_matrix(y_holdout, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrBr",
            xticklabels=data.target_names,
            yticklabels=data.target_names, ax=ax)
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusión – Hold-out (20%)")
plt.tight_layout()
plt.show()

## Parte 4 – Respuestas modelo a las preguntas de reflexión

**1. ¿Por qué en este dataset es especialmente importante usar StratifiedKFold?**  
Porque hay un desbalance (aprox. 37% malignant / 63% benign). Un K-Fold simple podría generar folds de validación con muy pocos (o ningún) casos malignos, haciendo que métricas como F1 o recall sean inestables o indefinidas. Stratified garantiza la misma proporción en cada fold.

**2. ¿Qué diferencia observaste entre el accuracy de una sola partición y el promedio de Stratified K-Fold?**  
El accuracy de una sola partición puede subir o bajar varios puntos porcentuales según la semilla. El promedio de Stratified K-Fold es más estable y representa mejor el desempeño esperado en datos nuevos. La desviación estándar de los folds nos da una idea de la robustez del modelo.

**3. ¿Por qué F1 puede ser mejor que accuracy en un problema de diagnóstico médico?**  
Porque el accuracy puede ser engañosamente alto si el modelo predice siempre la clase mayoritaria (benign). En medicina nos importa especialmente detectar los casos malignos (recall de la clase positiva) y no generar demasiados falsos positivos. F1 equilibra precisión y recall, y es más sensible al desempeño en la clase minoritaria/crítica.

**4. Principio de visualización aplicado**  
(Ejemplo de respuesta aceptable)  
- Color con significado (no decorativo): usé marrón para malignant y dorado para benign.  
- Título que comunica el insight: "Tumores malignos tienden a mayor radio y más puntos cóncavos".  
- Principio de similitud (Gestalt): los puntos del mismo color se agrupan visualmente.

**5. Visualización para un médico no técnico**  
Una matriz de confusión simple (o un gráfico de barras de Verdaderos Positivos / Falsos Negativos) acompañada de una frase clara:  
"De cada 100 pacientes con tumor maligno, el modelo detecta correctamente a X y se equivoca en Y".  
También un scatter o boxplot de la variable más discriminativa puede ayudar a explicar por qué el modelo funciona.